In [1]:
library(tidyverse)

Warning message:
“package ‘tidyr’ was built under R version 4.4.3”
Warning message:
“package ‘readr’ was built under R version 4.4.3”
Warning message:
“package ‘dplyr’ was built under R version 4.4.3”
Warning message:
“package ‘stringr’ was built under R version 4.4.3”
Warning message:
“package ‘forcats’ was built under R version 4.4.3”
Warning message:
“package ‘lubridate’ was built under R version 4.4.3”
── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   4.0.2     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.4     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


In [2]:
#Import individuals from bamfiles
inds<-Sys.glob("/mnt/beegfs/scratch/jbos/samfiles/*bam")

In [3]:
inds<-sapply(strsplit(as.character(inds), "files/"), `[`, 2)
inds<-sapply(strsplit(as.character(inds), "[.]"), `[`, 1)
inds<-as.data.frame(inds)
colnames(inds)<-'Sample_ID'

In [4]:
#Import site locations with lat/lons
sites <- read.csv('all_Atenuis_sites_FIXED.csv')
metadat<-read.csv('metadata_shoredist.csv')

In [5]:
nrow(metadat)

[1] 648

In [6]:
metadat<-left_join(inds,metadat)

Joining with `by = join_by(Sample_ID)`


In [7]:
nrow(metadat)
colnames(metadat)

[1] 652

[1] "Sample_ID"    "X"            "Unnamed..0"   "Time"         "Day"         
 [6] "Month"        "Year"         "Depth_ft"     "Unnamed..6"   "Unnamed..7"  
[11] "hour"         "minute"       "dt"           "lat"          "lon"         
[16] "name"         "DisplayColor" "Distance"     "ele"          "time"        
[21] "SHOREDIST"

In [49]:
cols<-c('sample_name','sample_title','bioproject_accession','organism','isolate','breed','host','isolation_source','collection_date','geo_loc_name','tissue','age','altitude','biomaterial_provider','collected_by','depth','dev_stage','env_broad_scale','host_tissue_sampled','identified_by','lat_lon','sex','specimen_voucher','temp','description')

In [43]:
cols

[1] "sample_name"          "sample_title"         "bioproject_accession"
 [4] "organism"             "isolate"              "collection_date"     
 [7] "geo_loc_name"         "collected_by"         "depth"               
[10] "dev_stage"            "lat_lon"              "library_ID"          
[13] "title"                "library_strategy"     "library_source"      
[16] "library_selection"    "library_layout"       "platform"            
[19] "instrument_model"     "design_description"   "filetype"            
[22] "filename"

In [10]:
ncbi<-as.data.frame(matrix(NA,ncol=length(cols),nrow=nrow(metadat)))
colnames(ncbi)<-cols

In [37]:
ncbi$organism<-'Acropora tenuis'

In [12]:
ncbi$tissue<-'skeleton_and_polyps'

In [13]:
ncbi$dev_stage<-'adult'

In [14]:
ncbi$collected_by<-'LisaMcManus'

In [15]:
colnames(metadat)
colnames(sites)
colnames(ncbi)

[1] "Sample_ID"    "X"            "Unnamed..0"   "Time"         "Day"         
 [6] "Month"        "Year"         "Depth_ft"     "Unnamed..6"   "Unnamed..7"  
[11] "hour"         "minute"       "dt"           "lat"          "lon"         
[16] "name"         "DisplayColor" "Distance"     "ele"          "time"        
[21] "SHOREDIST"

[1] "LONGITUDE" "LATITUDE"  "Name"

[1] "sample_name"          "sample_title"         "bioproject_accession"
 [4] "organism"             "isolate"              "breed"               
 [7] "host"                 "isolation_source"     "collection_date"     
[10] "geo_loc_name"         "tissue"               "age"                 
[13] "altitude"             "biomaterial_provider" "collected_by"        
[16] "depth"                "dev_stage"            "env_broad_scale"     
[19] "host_tissue_sampled"  "identified_by"        "lat_lon"             
[22] "sex"                  "specimen_voucher"     "temp"                
[25] "description"

In [16]:
ncbi$sample_name<-metadat$Sample_ID

In [17]:
ncbi$depth<-metadat$Depth_ft
for (j in 1:nrow(metadat)){
if (is.na(ncbi$depth[j])==FALSE){
    ncbi$depth[j]<-paste0(ncbi$depth[j],'ft')
    }else{
        ncbi$depth[j]<-ncbi$depth[j]
        }   
}

In [18]:
ncbi$site<-sapply(strsplit(as.character(ncbi$sample_name), "_"), `[`, 1)

In [19]:
colnames(sites)<-c('longitude','latitude','site')
ncbi<-left_join(ncbi,sites,by='site')

In [20]:
ncbi$lat2<-metadat$lat
ncbi$lon2<-metadat$lon

In [21]:
for (j in 1:nrow(ncbi)){
if (is.na(ncbi$lat2[j])==FALSE){
    ncbi$latitude[j]<-ncbi$lat2[j]
    }else{
        ncbi$latitude[j]<-ncbi$latitude[j]
        }   
}

In [22]:
for (j in 1:nrow(ncbi)){
if (is.na(ncbi$lon2[j])==FALSE){
    ncbi$longitude[j]<-ncbi$lon2[j]
    }else{
        ncbi$longitude[j]<-ncbi$longitude[j]
        }   
}

In [23]:
ncbi$lat_lon<-paste(ncbi$latitude,ncbi$longitude,sep="_")

In [24]:
table(is.na(metadat$dt))
metadat$site<-sapply(strsplit(as.character(metadat$Sample_ID), "_"), `[`, 1)


FALSE  TRUE 
  616    36 

In [26]:
#Match sampling dates for samples missing from metadat (all sites were sampled in single day)
for (i in 1:nrow(metadat)){
    if (is.na(metadat$dt[i])){
        s<-metadat[i,]$site
        metadat[i,]$dt<-metadat[metadat$site==s,]$dt[1]
        print(s)
        } else{
        metadat[i,]$dt<-metadat[i,]$dt
        }
    }

[1] "CEB15"
[1] "CEB15"
[1] "CEB15"
[1] "CEB15"
[1] "CEB15"


In [27]:
#Still missing CEB15 date. 
table(is.na(metadat$dt))


FALSE  TRUE 
  647     5 

In [28]:
#Got data from photo metadata. January 20 2016
metadat[metadat$site=='CEB15',]$dt<-'2016-01-20 12:00:00+08:00'

In [29]:
ncbi$collection_date<-sapply(strsplit(as.character(metadat$dt), " "), `[`, 1)

In [30]:
ncbi$island<-substr(ncbi$site,1,3)

In [31]:
table(ncbi$island)


CEB LEY 
496 156 

In [32]:
for (j in 1:nrow(metadat)){
if (ncbi$island[j]=='CEB'){
    ncbi$geo_loc_name[j]<-'Philippines:Cebu'
    }else{
        ncbi$geo_loc_name[j]<-'Philippines:Leyte'
        }   
}

In [33]:
table(ncbi$geo_loc_name)


 Philippines:Cebu Philippines:Leyte 
              496               156 

In [34]:
ncbi<-ncbi[,(colnames(ncbi) %in% cols)]

In [35]:
ncbi$isolate<-ncbi$sample_name

In [40]:
ncbi$bioproject_accession<-""

In [41]:
write.table(ncbi,'Acropora_tenuis_NCBI.txt',sep = "\t", row.names = FALSE,quote=FALSE)

In [56]:
sra_cols<-c('sample_name','library_ID','title','library_strategy','library_source','library_selection','library_layout','llatform','instrument_model','design_description','filetype','filename','filename2')

In [57]:
ncbi_sra<-as.data.frame(matrix(NA,ncol=length(sra_cols),nrow=nrow(inds)))
colnames(ncbi_sra)<-sra_cols

In [58]:
filenames<-Sys.glob("/mnt/beegfs/home/jbos/merged_lanes/*F.fq.gz")

In [59]:
filenames_R<-Sys.glob("/mnt/beegfs/home/jbos/merged_lanes/*R.fq.gz")

In [60]:
filenames<-sapply(strsplit(as.character(filenames), "lanes/"), `[`, 2)
filenames_R<-sapply(strsplit(as.character(filenames_R), "lanes/"), `[`, 2)

In [61]:
samp_names<-sapply(strsplit(as.character(filenames), ".F"), `[`, 1)

In [62]:
ncbi_sra$sample_name<-samp_names
ncbi_sra$library_ID<-samp_names
ncbi_sra$title<-'ddRadSeq of Acropora tenuis Philippines'
ncbi_sra$library_strategy<-'RAD-Seq'
ncbi_sra$library_source<-'Genomic'
ncbi_sra$library_selection<-'Restriction digest'
ncbi_sra$library_layout<-'paired'
ncbi_sra$platform<-'ILLUMINA'
ncbi_sra$instrument_model<-'Illumina HiSeq 4000'
ncbi_sra$design_description<-'enzymes SphI and EcoRI and targeted fragment sizes of 500-600 basepairs'
ncbi_sra$filetype<-'fastq'

In [63]:
ncbi_sra$filename<-filenames
ncbi_sra$filename2<-filenames_R

In [64]:
ncbi_sra

sample_name,library_ID,title,library_strategy,library_source,library_selection,library_layout,llatform,instrument_model,design_description,filetype,filename,filename2,platform
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<lgl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
CEB01_T01,CEB01_T01,ddRadSeq of Acropora tenuis Philippines,RAD-Seq,Genomic,Restriction digest,paired,NA,Illumina HiSeq 4000,enzymes SphI and EcoRI and targeted fragment sizes of 500-600 basepairs,fastq,CEB01_T01.F.fq.gz,CEB01_T01.R.fq.gz,ILLUMINA
CEB01_T02,CEB01_T02,ddRadSeq of Acropora tenuis Philippines,RAD-Seq,Genomic,Restriction digest,paired,NA,Illumina HiSeq 4000,enzymes SphI and EcoRI and targeted fragment sizes of 500-600 basepairs,fastq,CEB01_T02.F.fq.gz,CEB01_T02.R.fq.gz,ILLUMINA
CEB01_T03,CEB01_T03,ddRadSeq of Acropora tenuis Philippines,RAD-Seq,Genomic,Restriction digest,paired,NA,Illumina HiSeq 4000,enzymes SphI and EcoRI and targeted fragment sizes of 500-600 basepairs,fastq,CEB01_T03.F.fq.gz,CEB01_T03.R.fq.gz,ILLUMINA
CEB01_T04,CEB01_T04,ddRadSeq of Acropora tenuis Philippines,RAD-Seq,Genomic,Restriction digest,paired,NA,Illumina HiSeq 4000,enzymes SphI and EcoRI and targeted fragment sizes of 500-600 basepairs,fastq,CEB01_T04.F.fq.gz,CEB01_T04.R.fq.gz,ILLUMINA
CEB01_T05,CEB01_T05,ddRadSeq of Acropora tenuis Philippines,RAD-Seq,Genomic,Restriction digest,paired,NA,Illumina HiSeq 4000,enzymes SphI and EcoRI and targeted fragment sizes of 500-600 basepairs,fastq,CEB01_T05.F.fq.gz,CEB01_T05.R.fq.gz,ILLUMINA
CEB01_T06,CEB01_T06,ddRadSeq of Acropora tenuis Philippines,RAD-Seq,Genomic,Restriction digest,paired,NA,Illumina HiSeq 4000,enzymes SphI and EcoRI and targeted fragment sizes of 500-600 basepairs,fastq,CEB01_T06.F.fq.gz,CEB01_T06.R.fq.gz,ILLUMINA
CEB01_T07,CEB01_T07,ddRadSeq of Acropora tenuis Philippines,RAD-Seq,Genomic,Restriction digest,paired,NA,Illumina HiSeq 4000,enzymes SphI and EcoRI and targeted fragment sizes of 500-600 basepairs,fastq,CEB01_T07.F.fq.gz,CEB01_T07.R.fq.gz,ILLUMINA
CEB01_T08,CEB01_T08,ddRadSeq of Acropora tenuis Philippines,RAD-Seq,Genomic,Restriction digest,paired,NA,Illumina HiSeq 4000,enzymes SphI and EcoRI and targeted fragment sizes of 500-600 basepairs,fastq,CEB01_T08.F.fq.gz,CEB01_T08.R.fq.gz,ILLUMINA
CEB01_T09,CEB01_T09,ddRadSeq of Acropora tenuis Philippines,RAD-Seq,Genomic,Restriction digest,paired,NA,Illumina HiSeq 4000,enzymes SphI and EcoRI and targeted fragment sizes of 500-600 basepairs,fastq,CEB01_T09.F.fq.gz,CEB01_T09.R.fq.gz,ILLUMINA


In [65]:
write.table(ncbi_sra,'Acropora_tenuis_SRA.txt',sep = "\t", row.names = FALSE,quote=FALSE)